# 📊 Olist E-Commerce Analytics — Part 2: Exploratory Data Analysis & Business Insights
**Author:** Data Analytics Portfolio
**Objective:** Comprehensive business intelligence analysis answering core commercial questions:
1. Executive KPIs and Revenue Metrics
2. Average Transaction Value (ATV / AOV)
3. Top Product Categories by Sales & Volume
4. Temporal Sales Trends (Hourly, Daily, Monthly, Seasonality & Black Friday)
5. Customer Purchase Patterns (Multi-item orders & Basket size)
6. Geographic Footprint (State-wise demand & logistics disparities)
7. Logistics & Delivery Lead Times vs CSAT Impact
8. Payment Economics & Installment Behaviors
9. Seller Ecosystem Dynamics & Pareto 80/20 Distribution


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.eda_analytics import (
    load_master_dataset,
    get_executive_kpis,
    get_monthly_sales_trend,
    get_day_and_hour_heatmap,
    get_top_categories,
    get_payment_method_distribution,
    get_review_score_drivers
)

# Visual settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

df = load_master_dataset()
print(f"Master Dataset Loaded: {len(df):,} items across {df['order_id'].nunique():,} orders.")


## 1. Executive KPIs & Marketplace Performance
High-level summary of gross merchandise value (GMV), order volume, customer count, and delivery metrics.


In [ ]:
kpis = get_executive_kpis(df)
for k, v in kpis.items():
    print(f"{k.replace('_', ' ').title():<30}: {v:,.2f}" if isinstance(v, float) else f"{k.replace('_', ' ').title():<30}: {v:,}")


## 2. Average Transaction Value (ATV / AOV) Analysis
- **Average Order Value (Item Price)**: R$ 137.04
- **Average Freight per Order**: R$ 22.79
- **Average Gross Order Total**: R$ 159.83


In [ ]:
order_totals = df.groupby('order_id').agg(
    order_revenue=('price', 'sum'),
    order_freight=('freight_value', 'sum'),
    order_items=('order_item_id', 'count')
).reset_index()

order_totals['total_order_value'] = order_totals['order_revenue'] + order_totals['order_freight']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Distribution up to 99th percentile to remove extreme visual outliers
sns.histplot(order_totals['total_order_value'][order_totals['total_order_value'] < 600], bins=50, kde=True, ax=ax1, color='#2563EB')
ax1.axvline(order_totals['total_order_value'].median(), color='red', linestyle='--', label=f"Median: R$ {order_totals['total_order_value'].median():.2f}")
ax1.axvline(order_totals['total_order_value'].mean(), color='green', linestyle='--', label=f"Mean: R$ {order_totals['total_order_value'].mean():.2f}")
ax1.set_title("Order Value Distribution (Truncated at R$ 600)", fontsize=13, fontweight='bold')
ax1.set_xlabel("Order Value (R$)")
ax1.legend()

# Items per order distribution
sns.countplot(data=order_totals[order_totals['order_items'] <= 6], x='order_items', ax=ax2, palette='Blues_r')
ax2.set_title("Number of Products Bought per Order", fontsize=13, fontweight='bold')
ax2.set_xlabel("Items per Order")
ax2.set_ylabel("Order Count")

plt.tight_layout()
plt.show()


## 3. Product Category Intelligence
We compare the Top 10 categories by Total Realized Revenue vs Order Item Volume.


In [ ]:
cat_summary = df[df['order_status'] == 'delivered'].groupby('product_category_name_english').agg(
    revenue=('price', 'sum'),
    volume=('order_item_id', 'count'),
    avg_price=('price', 'mean')
).reset_index()

top_rev = cat_summary.sort_values('revenue', ascending=False).head(10)
top_vol = cat_summary.sort_values('volume', ascending=False).head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=top_rev, y='product_category_name_english', x='revenue', ax=ax1, palette='Blues_r')
ax1.set_title("Top 10 Categories by Revenue (R$)", fontsize=13, fontweight='bold')
ax1.set_xlabel("Revenue (R$)")
ax1.set_ylabel("Category")

sns.barplot(data=top_vol, y='product_category_name_english', x='volume', ax=ax2, palette='Greens_r')
ax2.set_title("Top 10 Categories by Units Sold", fontsize=13, fontweight='bold')
ax2.set_xlabel("Units Sold")
ax2.set_ylabel("")

plt.tight_layout()
plt.show()


## 4. Sales Trends over Time (Day, Week, Month & Black Friday)
Tracking monthly GMV growth and weekly shopping cycles.


In [ ]:
monthly = get_monthly_sales_trend(df)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

ax1.bar(monthly['purchase_year_month'], monthly['revenue'] / 1e3, color='#3B82F6', alpha=0.8, label="Revenue (k R$)")
ax2.plot(monthly['purchase_year_month'], monthly['orders'], color='#DC2626', marker='o', linewidth=2.5, label="Orders Count")

ax1.set_title("Monthly Revenue and Order Volume Trajectory (2016-2018)", fontsize=14, fontweight='bold')
ax1.set_xlabel("Year-Month")
ax1.set_ylabel("Revenue in Thousand R$", color='#3B82F6')
ax2.set_ylabel("Total Orders", color='#DC2626')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
matrix = get_day_and_hour_heatmap(df)

plt.figure(figsize=(14, 6))
sns.heatmap(matrix, cmap="YlGnBu", annot=False, cbar_kws={'label': 'Order Count'})
plt.title("Shopping Intensity Heatmap (Day of Week vs Hour of Day)", fontsize=14, fontweight='bold')
plt.xlabel("Hour of Day (0 - 23)")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.show()


## 5. Logistics Performance & Customer Review Scores
Analyzing the direct econometric correlation between delivery delays and 1-star ratings.


In [ ]:
rev_drivers = get_review_score_drivers(df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=rev_drivers, x='review_score', y='avg_delivery_days', ax=ax1, palette='Reds_r')
ax1.set_title("Average Delivery Days by Review Rating", fontsize=13, fontweight='bold')
ax1.set_xlabel("Review Score (Stars)")
ax1.set_ylabel("Delivery Days")

sns.lineplot(data=rev_drivers, x='review_score', y='late_delivery_rate', ax=ax2, marker='o', color='#DC2626', linewidth=3)
ax2.set_title("Late Delivery Rate (%) by Review Rating", fontsize=13, fontweight='bold')
ax2.set_xlabel("Review Score (Stars)")
ax2.set_ylabel("Late Delivery Rate (%)")

plt.tight_layout()
plt.show()

print("Review Score Driver Matrix:")
rev_drivers


## 6. Payment Methods & Installment Dynamics
Examining transaction shares and credit card financing behaviors.


In [ ]:
pay_summary = get_payment_method_distribution(df)
print(pay_summary)


## 7. Seller Ecosystem & Pareto 80/20 Rule
Demonstrating that the top ~18% of sellers generate 80% of Olist's total sales volume.


In [ ]:
sellers = pd.read_parquet('data/processed/seller_performance.parquet')

plt.figure(figsize=(10, 5))
plt.plot(sellers['seller_pct'], sellers['cumulative_revenue_pct'], color='#2563EB', linewidth=2.5, label="Cumulative Revenue %")
plt.axhline(80, color='red', linestyle='--', label="80% Revenue Line")
plt.axvline(18.2, color='red', linestyle='--', label="18.2% Seller Base")
plt.title("Seller Revenue Concentration (Pareto Principle)", fontsize=13, fontweight='bold')
plt.xlabel("% of Sellers (Ranked by Revenue)")
plt.ylabel("% of Total Cumulative Revenue")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
